# Urban Roaming VQA: Walk Trace Analysis

Analyze and visualize walk traces from the urbanroamvqa pipeline. This notebook explores agent navigation patterns, directional preferences, reasoning text, and spatial coverage across multiple simulated urban walks.

## Section 1: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try to import optional visualization libraries
try:
    import folium
    from folium.plugins import HeatMap
    FOLIUM_AVAILABLE = True
except ImportError:
    FOLIUM_AVAILABLE = False
    print("Warning: folium not available. Maps will use matplotlib scatter plots instead.")

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print("Warning: wordcloud not available. Using bar charts for text analysis instead.")

from IPython.display import display, HTML
import matplotlib.patches as mpatches

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Imports successful!")

### Configuration

Update the `TRACE_PARQUET_PATH` to point to your trace output file from the urbanroamvqa pipeline.

In [ ]:
# Path to trace parquet file - UPDATE THIS
TRACE_PARQUET_PATH = "/path/to/your/trace_output.parquet"

# Optional: limit number of walks for visualization (None = all)
WALKS_FOR_MAP = 20

# Output directory for maps
OUTPUT_DIR = Path("/share/pierson/matt/mllmsci/notebooks/roaming/")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Trace path: {TRACE_PARQUET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

### Load Data

In [ ]:
# Load trace data
df = pd.read_parquet(TRACE_PARQUET_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst few rows:")
display(df.head(10))

### Basic Statistics

In [ ]:
# Calculate basic stats
n_walks = df['walk_id'].nunique()
n_steps = len(df)
avg_steps_per_walk = n_steps / n_walks

print(f"Total number of walks: {n_walks}")
print(f"Total number of steps: {n_steps}")
print(f"Average steps per walk: {avg_steps_per_walk:.2f}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nData coverage:")
print(f"Walks with termination reason: {df['termination_reason'].notna().sum()} / {n_walks}")
print(f"Walks with coordinates: {df[['lat', 'lon']].notna().all(axis=1).sum()} / {n_steps}")

## Section 2: Walk Length Distribution

In [ ]:
# Calculate walk lengths
walk_lengths = df.groupby('walk_id').size()

print(f"Walk Length Statistics:")
print(f"  Mean:   {walk_lengths.mean():.2f}")
print(f"  Median: {walk_lengths.median():.2f}")
print(f"  Std:    {walk_lengths.std():.2f}")
print(f"  Min:    {walk_lengths.min()}")
print(f"  Max:    {walk_lengths.max()}")
print(f"  25th percentile: {walk_lengths.quantile(0.25):.2f}")
print(f"  75th percentile: {walk_lengths.quantile(0.75):.2f}")

In [ ]:
# Visualize walk length distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(walk_lengths, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Walk Length (steps)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Walk Lengths')
axes[0, 0].axvline(walk_lengths.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {walk_lengths.mean():.1f}')
axes[0, 0].axvline(walk_lengths.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {walk_lengths.median():.1f}')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Box plot
axes[0, 1].boxplot(walk_lengths, vert=True)
axes[0, 1].set_ylabel('Walk Length (steps)')
axes[0, 1].set_title('Walk Length Box Plot')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# CDF
sorted_lengths = np.sort(walk_lengths)
cdf = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1, 0].plot(sorted_lengths, cdf, marker='o', linestyle='-', markersize=3, alpha=0.7)
axes[1, 0].set_xlabel('Walk Length (steps)')
axes[1, 0].set_ylabel('Cumulative Probability')
axes[1, 0].set_title('Cumulative Distribution of Walk Lengths')
axes[1, 0].grid(True, alpha=0.3)

# Walk lengths by walk (sorted)
walk_lengths_sorted = walk_lengths.sort_values(ascending=False)
if len(walk_lengths_sorted) <= 50:
    axes[1, 1].bar(range(len(walk_lengths_sorted)), walk_lengths_sorted.values, alpha=0.7, color='coral')
    axes[1, 1].set_xlabel('Walk (ranked by length)')
    axes[1, 1].set_ylabel('Length (steps)')
    axes[1, 1].set_title('Individual Walk Lengths (sorted)')
else:
    axes[1, 1].scatter(range(len(walk_lengths_sorted)), walk_lengths_sorted.values, alpha=0.5, s=20)
    axes[1, 1].set_xlabel('Walk (ranked by length)')
    axes[1, 1].set_ylabel('Length (steps)')
    axes[1, 1].set_title(f'Individual Walk Lengths (sorted, n={len(walk_lengths_sorted)})')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'walk_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: walk_lengths.png")

## Section 3: Spatial Coverage Map

In [ ]:
# Prepare data for mapping
df_with_coords = df[['walk_id', 'step_n', 'lat', 'lon']].dropna(subset=['lat', 'lon'])

if len(df_with_coords) == 0:
    print("No coordinate data available for mapping.")
else:
    # Calculate map center
    center_lat = df_with_coords['lat'].median()
    center_lon = df_with_coords['lon'].median()
    
    print(f"Map center: ({center_lat:.6f}, {center_lon:.6f})")
    print(f"Lat range: {df_with_coords['lat'].min():.6f} to {df_with_coords['lat'].max():.6f}")
    print(f"Lon range: {df_with_coords['lon'].min():.6f} to {df_with_coords['lon'].max():.6f}")

In [ ]:
# Create spatial coverage map with folium if available
if FOLIUM_AVAILABLE and len(df_with_coords) > 0:
    # Get unique walks for mapping (limit to first N for readability)
    unique_walks = df['walk_id'].unique()
    if WALKS_FOR_MAP is not None:
        unique_walks = unique_walks[:WALKS_FOR_MAP]
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=14,
        tiles='OpenStreetMap'
    )
    
    # Color palette for walks
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_walks)))
    color_hex = [f'#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}' for c in colors]
    
    # Add walk traces
    for walk_idx, walk_id in enumerate(unique_walks):
        walk_data = df[df['walk_id'] == walk_id][['step_n', 'lat', 'lon']].dropna(subset=['lat', 'lon'])
        walk_data = walk_data.sort_values('step_n')
        
        if len(walk_data) > 1:
            # Create polyline for walk
            coords = list(zip(walk_data['lat'], walk_data['lon']))
            folium.PolyLine(
                coords,
                color=color_hex[walk_idx % len(color_hex)],
                weight=2,
                opacity=0.7,
                popup=f"Walk {walk_id}"
            ).add_to(m)
            
            # Add markers for start and end
            folium.CircleMarker(
                location=[walk_data.iloc[0]['lat'], walk_data.iloc[0]['lon']],
                radius=5,
                color=color_hex[walk_idx % len(color_hex)],
                fill=True,
                fillOpacity=0.8,
                popup=f"Walk {walk_id} Start"
            ).add_to(m)
    
    # Save and display
    map_path = OUTPUT_DIR / 'spatial_coverage_map.html'
    m.save(str(map_path))
    print(f"Saved: spatial_coverage_map.html")
    display(m)
    
elif not FOLIUM_AVAILABLE:
    print("Folium not available. Creating scatter plot instead...")
    plt.figure(figsize=(12, 10))
    
    # Create scatter plot colored by walk
    unique_walks = df['walk_id'].unique()
    if WALKS_FOR_MAP is not None:
        unique_walks = unique_walks[:WALKS_FOR_MAP]
    
    for walk_id in unique_walks:
        walk_data = df[df['walk_id'] == walk_id][['lat', 'lon']].dropna()
        plt.scatter(walk_data['lon'], walk_data['lat'], s=30, alpha=0.6, label=walk_id[:8])
    
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title(f'Spatial Coverage ({len(unique_walks)} walks shown)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'spatial_coverage_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: spatial_coverage_scatter.png")

## Section 4: Directional Preferences

In [ ]:
# Analyze face choices
face_counts = df['face_chosen'].value_counts()
face_probs = df['face_chosen'].value_counts(normalize=True)

print("Face Choice Distribution:")
print(face_counts)
print("\nFace Choice Probabilities:")
print(face_probs)

# Calculate directional bias (are certain directions chosen more often?)
print("\nDirectional Bias:")
print(f"Forward (F) dominance: {face_probs.get('F', 0):.2%}")
print(f"Right (R) dominance: {face_probs.get('R', 0):.2%}")
print(f"Left (L) dominance: {face_probs.get('L', 0):.2%}")
print(f"Backward (B) dominance: {face_probs.get('B', 0):.2%}")

In [ ]:
# Visualize directional preferences
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Bar chart of face choices
face_counts = df['face_chosen'].value_counts().sort_index()
axes[0, 0].bar(face_counts.index, face_counts.values, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
axes[0, 0].set_xlabel('Face Direction')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Face Choice Distribution')
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(face_counts.values):
    axes[0, 0].text(i, v + max(face_counts.values) * 0.01, str(v), ha='center', va='bottom')

# Pie chart
colors_pie = ['skyblue', 'lightcoral', 'lightgreen', 'gold']
axes[0, 1].pie(face_counts.values, labels=face_counts.index, autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[0, 1].set_title('Face Choice Proportions')

# Bearing distribution (if available)
if 'bearing_deg' in df.columns:
    bearing_data = df['bearing_deg'].dropna()
    if len(bearing_data) > 0:
        # Bin bearings into compass directions
        bins = [0, 45, 90, 135, 180, 225, 270, 315, 360]
        labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
        bearing_binned = pd.cut(bearing_data, bins=bins, labels=labels, right=False, include_lowest=True)
        bearing_counts = bearing_binned.value_counts().reindex(labels)
        
        # Bar chart
        axes[1, 0].bar(bearing_counts.index, bearing_counts.values, color='teal', alpha=0.7)
        axes[1, 0].set_xlabel('Direction')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].set_title('Movement Bearing Distribution (8 Compass Directions)')
        axes[1, 0].grid(True, alpha=0.3, axis='y')
        
        # Polar/Rose diagram
        ax_polar = plt.subplot(2, 2, 4, projection='polar')
        angles = np.array([0, 45, 90, 135, 180, 225, 270, 315]) * np.pi / 180
        values = bearing_counts.values
        angles_plot = np.concatenate((angles, [angles[0]]))
        values_plot = np.concatenate((values, [values[0]]))
        ax_polar.plot(angles_plot, values_plot, 'o-', linewidth=2, color='teal')
        ax_polar.fill(angles_plot, values_plot, alpha=0.25, color='teal')
        ax_polar.set_theta_zero_location('N')
        ax_polar.set_theta_direction(-1)
        ax_polar.set_xticks(angles)
        ax_polar.set_xticklabels(labels)
        ax_polar.set_title('Movement Bearing Rose Diagram', pad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'directional_preferences.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: directional_preferences.png")

In [ ]:
# Analyze arrival_face vs face_chosen (does the agent go straight, turn, etc.?)
if 'arrival_face' in df.columns:
    df_with_faces = df[['arrival_face', 'face_chosen']].dropna()
    
    if len(df_with_faces) > 0:
        # Cross-tabulation
        cross_tab = pd.crosstab(df_with_faces['arrival_face'], df_with_faces['face_chosen'], margins=True)
        print("Arrival Face vs Chosen Face (cross-tabulation):")
        display(cross_tab)
        
        # Calculate forward bias (choosing F when arriving with F)
        straight_choices = df_with_faces[df_with_faces['arrival_face'] == df_with_faces['face_chosen']]
        straight_pct = len(straight_choices) / len(df_with_faces) * 100
        print(f"\nAgent goes straight (same direction as arrival): {straight_pct:.1f}%")

## Section 5: Distance & Spatial Metrics

In [ ]:
# Visualize distance metrics
if len(distance_data) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Histogram of per-step distances
    axes[0, 0].hist(distance_data, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].set_xlabel('Distance (meters)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Per-Step Distance Distribution')
    axes[0, 0].axvline(distance_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {distance_data.mean():.1f}m')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Box plot
    axes[0, 1].boxplot(distance_data, vert=True)
    axes[0, 1].set_ylabel('Distance (meters)')
    axes[0, 1].set_title('Per-Step Distance Box Plot')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Total distance per walk (bar chart, sorted)
    walk_distances_sorted = walk_distances.sort_values(ascending=False)
    if len(walk_distances_sorted) <= 50:
        axes[1, 0].bar(range(len(walk_distances_sorted)), walk_distances_sorted.values, alpha=0.7, color='coral')
        axes[1, 0].set_xlabel('Walk (ranked by distance)')
        axes[1, 0].set_ylabel('Total Distance (meters)')
        axes[1, 0].set_title('Total Distance per Walk (sorted)')
    else:
        axes[1, 0].scatter(range(len(walk_distances_sorted)), walk_distances_sorted.values, alpha=0.5, s=20)
        axes[1, 0].set_xlabel('Walk (ranked by distance)')
        axes[1, 0].set_ylabel('Total Distance (meters)')
        axes[1, 0].set_title(f'Total Distance per Walk (sorted, n={len(walk_distances_sorted)})')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Distance vs walk length
    walk_data_merged = pd.DataFrame({
        'walk_length': walk_lengths,
        'total_distance': walk_distances
    })
    axes[1, 1].scatter(walk_data_merged['walk_length'], walk_data_merged['total_distance'], alpha=0.6, s=50)
    axes[1, 1].set_xlabel('Walk Length (steps)')
    axes[1, 1].set_ylabel('Total Distance (meters)')
    axes[1, 1].set_title('Walk Length vs Total Distance')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(walk_data_merged['walk_length'], walk_data_merged['total_distance'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(walk_data_merged['walk_length'].min(), walk_data_merged['walk_length'].max(), 100)
    axes[1, 1].plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'distance_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: distance_metrics.png")
else:
    print("No distance data to visualize.")

In [ ]:
# Calculate spatial extent per walk (bounding box area and convex hull if scipy available)
def get_spatial_extent(coords):
    """Calculate bounding box area for a set of coordinates."""
    if len(coords) < 2:
        return 0
    lat_range = coords['lat'].max() - coords['lat'].min()
    lon_range = coords['lon'].max() - coords['lon'].min()
    # Very rough: treat as meters (0.01 degrees ~ 1.1 km)
    return lat_range * lon_range * 1.1 * 1.1 * 1e6  # approximate m^2

walk_extents = []
for walk_id in df['walk_id'].unique():
    walk_coords = df[df['walk_id'] == walk_id][['lat', 'lon']].dropna()
    if len(walk_coords) > 0:
        extent = get_spatial_extent(walk_coords)
        walk_extents.append({'walk_id': walk_id, 'extent': extent})

walk_extents_df = pd.DataFrame(walk_extents)
print(f"\nSpatial Extent Statistics (approx m²):")
print(f"  Mean:   {walk_extents_df['extent'].mean():.2f}")
print(f"  Median: {walk_extents_df['extent'].median():.2f}")
print(f"  Max:    {walk_extents_df['extent'].max():.2f}")

## Section 6: Termination Analysis

In [ ]:
# Analyze termination reasons
termination_counts = df['termination_reason'].value_counts()
print("Termination Reason Distribution:")
print(termination_counts)
print("\nTermination Reason Percentages:")
print((termination_counts / termination_counts.sum() * 100).round(2))

# Average walk length by termination reason
walk_with_termination = df[df['termination_reason'].notna()].groupby('walk_id').agg({
    'step_n': 'max',
    'termination_reason': 'first'
}).rename(columns={'step_n': 'walk_length'})

print("\nAverage Walk Length by Termination Reason:")
term_by_reason = walk_with_termination.groupby('termination_reason')['walk_length'].agg(['mean', 'count'])
print(term_by_reason)

In [ ]:
# Visualize termination analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of termination reasons
termination_counts = df['termination_reason'].value_counts()
axes[0].bar(termination_counts.index, termination_counts.values, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Termination Reason')
axes[0].set_ylabel('Count')
axes[0].set_title('Termination Reason Distribution')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(termination_counts.values):
    axes[0].text(i, v + max(termination_counts.values) * 0.01, str(v), ha='center', va='bottom')

# Box plot of walk length by termination reason
termination_data = []
termination_labels = []
for reason in termination_counts.index:
    walk_ids_for_reason = df[df['termination_reason'] == reason]['walk_id'].unique()
    lengths = [len(df[df['walk_id'] == wid]) for wid in walk_ids_for_reason]
    termination_data.append(lengths)
    termination_labels.append(reason)

if termination_data:
    bp = axes[1].boxplot(termination_data, labels=termination_labels, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    axes[1].set_ylabel('Walk Length (steps)')
    axes[1].set_title('Walk Length Distribution by Termination Reason')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'termination_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: termination_analysis.png")

## Section 7: Reasoning Text Analysis

In [ ]:
# Extract most common words from reasoning
import string

# Stop words
STOP_WORDS = set([
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
    'is', 'am', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'should', 'could', 'can', 'may', 'might', 'must',
    'i', 'me', 'my', 'we', 'you', 'he', 'she', 'it', 'they', 'them',
    'this', 'that', 'these', 'those', 'what', 'which', 'who', 'why', 'how',
    'as', 'if', 'than', 'because', 'while', 'when', 'where', 'there', 'here',
    'not', 'no', 'so', 'such', 'only', 'all', 'each', 'every', 'both', 'neither',
    'some', 'any', 'more', 'most', 'other', 'another', 'same', 'different',
    'very', 'too', 'just', 'also', 'even', 'now', 'then', 'before', 'after',
    'up', 'down', 'out', 'over', 'under', 'through', 'during', 'from', 'into',
    'about', 'above', 'below', 'between', 'among', 'around', 'near', 'against',
    'or', 'and', 'nor', 'yet', 'so'
])

# Extract words
words = []
for text in reasoning_data:
    # Convert to lowercase and split
    text_clean = text.lower()
    # Remove punctuation
    text_clean = text_clean.translate(str.maketrans('', '', string.punctuation))
    # Split into words
    text_words = text_clean.split()
    # Filter stop words and short words
    text_words = [w for w in text_words if len(w) > 2 and w not in STOP_WORDS]
    words.extend(text_words)

# Count frequencies
word_freq = Counter(words)
top_words = word_freq.most_common(20)

print("Top 20 most common words in reasoning:")
for word, count in top_words:
    print(f"  {word}: {count}")

In [ ]:
# Try to create wordcloud if available
if WORDCLOUD_AVAILABLE:
    # Combine all reasoning texts
    all_reasoning = ' '.join(reasoning_data)
    
    # Create wordcloud
    wc = WordCloud(width=1200, height=600, background_color='white', stopwords=STOP_WORDS).generate(all_reasoning)
    
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Reasoning Texts')
    plt.tight_layout(pad=0)
    plt.savefig(OUTPUT_DIR / 'reasoning_wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: reasoning_wordcloud.png")
else:
    print("WordCloud not available. Skipping wordcloud visualization.")

## Section 8: Single Walk Deep Dive

In [ ]:
# Find the longest walk
longest_walk_id = walk_lengths.idxmax()
longest_walk_length = walk_lengths.max()

print(f"Longest walk: {longest_walk_id}")
print(f"Length: {longest_walk_length} steps")

# Extract walk data
longest_walk_df = df[df['walk_id'] == longest_walk_id].sort_values('step_n').copy()

# Prepare display columns
display_cols = ['step_n', 'recording_id', 'arrival_face', 'face_chosen', 'distance_m', 'bearing_deg', 'termination_reason']
available_cols = [c for c in display_cols if c in longest_walk_df.columns]

print(f"\nWalk Summary Table (first 15 steps):")
display(longest_walk_df[available_cols].head(15))

In [ ]:
# Plot the longest walk trajectory
walk_coords = longest_walk_df[['lat', 'lon']].dropna()

if len(walk_coords) > 1:
    if FOLIUM_AVAILABLE:
        # Create folium map
        center_lat = walk_coords['lat'].median()
        center_lon = walk_coords['lon'].median()
        
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=16,
            tiles='OpenStreetMap'
        )
        
        # Add polyline for trajectory
        coords_list = list(zip(walk_coords['lat'], walk_coords['lon']))
        folium.PolyLine(coords_list, color='blue', weight=2, opacity=0.8).add_to(m)
        
        # Add numbered markers
        step_numbers = longest_walk_df[longest_walk_df[['lat', 'lon']].notna().all(axis=1)]['step_n'].values
        for idx, (lat, lon) in enumerate(coords_list):
            step_n = step_numbers[idx] if idx < len(step_numbers) else idx
            folium.CircleMarker(
                location=[lat, lon],
                radius=6,
                color='red' if idx == 0 else ('green' if idx == len(coords_list) - 1 else 'blue'),
                fill=True,
                fillOpacity=0.8,
                popup=f"Step {int(step_n)}"
            ).add_to(m)
        
        # Save and display
        map_path = OUTPUT_DIR / f'longest_walk_{longest_walk_id}_trajectory.html'
        m.save(str(map_path))
        print(f"Saved: longest_walk_{longest_walk_id}_trajectory.html")
        display(m)
    else:
        # Matplotlib fallback
        plt.figure(figsize=(12, 10))
        plt.plot(walk_coords['lon'], walk_coords['lat'], 'b-', alpha=0.6, linewidth=2, label='Path')
        plt.scatter(walk_coords['lon'].iloc[0], walk_coords['lat'].iloc[0], color='green', s=100, marker='o', label='Start', zorder=5)
        plt.scatter(walk_coords['lon'].iloc[-1], walk_coords['lat'].iloc[-1], color='red', s=100, marker='s', label='End', zorder=5)
        plt.scatter(walk_coords['lon'], walk_coords['lat'], color='blue', s=30, alpha=0.5, label='Steps')
        
        # Add step numbers
        step_numbers = longest_walk_df[longest_walk_df[['lat', 'lon']].notna().all(axis=1)]['step_n'].values
        for idx, (lat, lon) in enumerate(zip(walk_coords['lat'], walk_coords['lon'])):
            if idx % max(1, len(walk_coords) // 10) == 0:  # Label every N steps
                step_n = step_numbers[idx] if idx < len(step_numbers) else idx
                plt.text(lon, lat, str(int(step_n)), fontsize=8, ha='center')
        
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.title(f'Longest Walk Trajectory: {longest_walk_id} ({longest_walk_length} steps)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'longest_walk_{longest_walk_id}_trajectory.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: longest_walk_{longest_walk_id}_trajectory.png")
else:
    print("Insufficient coordinate data for trajectory visualization.")

In [ ]:
# Visualize walk statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distance per step
walk_distances_step = longest_walk_df.groupby('step_n')['distance_m'].sum()
axes[0, 0].bar(walk_distances_step.index, walk_distances_step.values, alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Step Number')
axes[0, 0].set_ylabel('Distance (meters)')
axes[0, 0].set_title('Distance per Step')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Cumulative distance
cumulative_distance = walk_distances_step.cumsum()
axes[0, 1].plot(cumulative_distance.index, cumulative_distance.values, marker='o', linewidth=2, markersize=4)
axes[0, 1].fill_between(cumulative_distance.index, cumulative_distance.values, alpha=0.3)
axes[0, 1].set_xlabel('Step Number')
axes[0, 1].set_ylabel('Cumulative Distance (meters)')
axes[0, 1].set_title('Cumulative Distance Traveled')
axes[0, 1].grid(True, alpha=0.3)

# Face choices per step
walk_faces = longest_walk_df['face_chosen'].value_counts()
axes[1, 0].bar(walk_faces.index, walk_faces.values, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
axes[1, 0].set_xlabel('Face Direction')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title(f'Face Choices in Walk {longest_walk_id}')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Reasoning length per step
reasoning_lengths = longest_walk_df['reasoning'].str.len().dropna()
axes[1, 1].plot(longest_walk_df['step_n'][:len(reasoning_lengths)], reasoning_lengths.values, marker='o', linestyle='-', linewidth=1, markersize=4, color='purple')
axes[1, 1].fill_between(longest_walk_df['step_n'][:len(reasoning_lengths)], reasoning_lengths.values, alpha=0.3, color='purple')
axes[1, 1].set_xlabel('Step Number')
axes[1, 1].set_ylabel('Reasoning Length (characters)')
axes[1, 1].set_title('Reasoning Text Length per Step')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'longest_walk_{longest_walk_id}_details.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: longest_walk_{longest_walk_id}_details.png")

## Summary & Next Steps

This notebook has analyzed urban roaming VQA walk traces across the following dimensions:

1. **Walk Length**: Explored the distribution of walk lengths (steps per walk)
2. **Spatial Coverage**: Visualized where agents explored using maps and scatter plots
3. **Directional Preferences**: Analyzed face choices (F/R/B/L) and bearing distributions
4. **Distance Metrics**: Examined per-step and total walk distances
5. **Termination Patterns**: Investigated why walks ended
6. **Reasoning Analysis**: Explored the VLM's reasoning text
7. **Deep Dive**: Detailed analysis of the longest walk

### Files Generated
All visualizations are saved to `/share/pierson/matt/mllmsci/notebooks/roaming/`:
- `walk_lengths.png` — Walk length distribution
- `spatial_coverage_map.html` or `spatial_coverage_scatter.png` — Spatial map
- `directional_preferences.png` — Face choice and bearing analysis
- `distance_metrics.png` — Distance statistics
- `termination_analysis.png` — Termination reason distribution
- `reasoning_analysis.png` — Reasoning text analysis
- `reasoning_wordcloud.png` — Word cloud (if available)
- `longest_walk_*_trajectory.html` or `.png` — Trajectory visualization
- `longest_walk_*_details.png` — Detailed walk statistics

### Suggested Next Steps
1. **Compare walks by type**: Are there patterns by image type, location, or time?
2. **Analyze decision quality**: Validate agent choices against human annotations
3. **Extract structured data**: Parse JSON responses for more detailed analysis
4. **Clustering**: Group similar walks by trajectory, distance, or reasoning patterns
5. **Temporal analysis**: If timestamps are available, explore time-based patterns